# Manufacturing Data Analysis EDA Skeleton

## Project Context

This notebook is a minimal starting point for later exploratory data analysis and project presentation work.

Current goal:

- understand equipment, line, shift, quality, and failure behavior from the current processed manufacturing dataset
- keep the analysis aligned with the current repository structure and contract tests

Important boundary:

- several KPI fields in this project, including `planned_production`, `actual_production`, `defect_rate`, `availability`, `performance`, `quality_rate`, and `oee`, are current project proxy / simulated metrics
- this notebook should support structured analysis, but not claim industrial-grade plant conclusions at this stage


## Analysis Questions

This notebook is designed to help answer questions like:

1. Which equipment or production lines show weaker OEE performance?
2. Do `Day`, `Evening`, and `Night` shifts show different KPI patterns?
3. How does machine failure relate to quality and OEE behavior?
4. Which equipment appears more often in failure-labelled records?
5. Which insights are supported by current proxy metrics, and which conclusions are still outside the scope of this project?


In [ ]:
from pathlib import Path

import pandas as pd

## Data Loading

Choose one processed CSV source.

- `manufacturing_data_processed.csv` is the current legacy reference
- `data/processed/manufacturing_data_processed_refactor.csv` is the current refactor snapshot

The default below uses the legacy processed CSV.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

legacy_processed_path = PROJECT_ROOT / 'manufacturing_data_processed.csv'
refactor_processed_path = PROJECT_ROOT / 'data' / 'processed' / 'manufacturing_data_processed_refactor.csv'

# Change this path if you want to inspect the refactor snapshot instead.
data_path = legacy_processed_path

print(f'Using dataset: {data_path}')
df = pd.read_csv(data_path)

print(f'Shape: {df.shape}')
display(df.head())

In [ ]:
df.columns.tolist()

## Dataset Overview

Start with the most basic checks before writing any interpretation.

This section answers a simple question first:

- What data do we actually have in hand before we compare equipment, lines, shifts, or failures?


### A. Basic Dataset Size And Key Column Presence

This check confirms whether the dataset has the expected scale and whether the main analysis fields are present.

In [ ]:
key_columns = ['production_time', 'equipment_id', 'oee', 'Machine failure']

basic_overview = {
    'row_count': len(df),
    'column_count': len(df.columns),
    'equipment_count': df['equipment_id'].nunique(),
    'production_line_count': df['production_line'].nunique(),
    'shift_categories': sorted(df['shift'].dropna().unique().tolist()),
    'key_columns_present': {column: (column in df.columns) for column in key_columns},
}

basic_overview

### B. Time Range Overview

This check tells us how wide the current analysis window is and how many calendar dates are covered.

In [ ]:
production_time_series = pd.to_datetime(df['production_time'])

time_overview = {
    'min_production_time': production_time_series.min(),
    'max_production_time': production_time_series.max(),
    'date_count': production_time_series.dt.date.nunique(),
}

time_overview

### C. Basic Missing Check

This is not a full data-quality audit. It only checks whether the most important fields are missing obvious values.

In [ ]:
missing_check_columns = [
    'production_time',
    'equipment_id',
    'production_line',
    'shift',
    'oee',
    'availability',
    'performance',
    'quality_rate',
    'Machine failure',
]

missing_summary = df[missing_check_columns].isna().sum().rename('missing_count').to_frame()
missing_summary['missing_ratio'] = (missing_summary['missing_count'] / len(df)).round(6)
missing_summary

### D. Basic KPI Summary

This gives a first numeric profile of the current KPI fields.

Important reminder:

- these are current project proxy / simulated metrics
- treat this as a local analysis baseline, not as industrial KPI certification


In [ ]:
kpi_summary = df[['oee', 'availability', 'performance', 'quality_rate']].describe().T
kpi_summary

### E. Short Interpretation Notes

Use this area to write 2 to 4 short observations after you inspect the outputs above.

Suggested prompts:

- Does the dataset size match what you expected?
- Are the key analysis columns present and mostly complete?
- Does the current time range look reasonable for a first analysis pass?
- Do the KPI summaries look broadly plausible under the current proxy logic?


## Equipment / Line KPI Analysis

Use this section to compare equipment and production lines.

This section answers the next practical question:

- Which equipment or lines look weaker under the current KPI rules, and where should deeper analysis start?


### A. Equipment-Level KPI Summary

This table is the first place to look for weaker equipment under the current proxy KPI design.

Suggested reading habit:

- first scan low `avg_oee`
- then check whether low `avg_oee` appears together with higher `failure_record_count` or `total_defect_count`


In [ ]:
equipment_kpi = (
    df.groupby('equipment_id', as_index=False)
    .agg(
        production_line=('production_line', 'first'),
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        total_actual_production=('actual_production', 'sum'),
        total_defect_count=('defect_count', 'sum'),
        failure_record_count=('Machine failure', 'sum'),
    )
    .sort_values(['avg_oee', 'failure_record_count', 'total_defect_count'], ascending=[True, False, False])
    .reset_index(drop=True)
)

equipment_kpi.head(10)

Interpretation prompt:

- Which equipment sits near the bottom of `avg_oee`?
- Are those same equipment IDs also showing more failure-labelled records or more total defects?
- Under the current proxy KPI rules, this supports prioritizing inspection targets, but it does not prove root cause.


### B. Production-Line KPI Summary

This table rolls the same logic up to production-line level so you can see whether weaker equipment also cluster into weaker lines.

In [ ]:
line_kpi = (
    df.groupby('production_line', as_index=False)
    .agg(
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        total_actual_production=('actual_production', 'sum'),
        total_defect_count=('defect_count', 'sum'),
        failure_record_count=('Machine failure', 'sum'),
    )
    .sort_values(['avg_oee', 'failure_record_count'], ascending=[True, False])
    .reset_index(drop=True)
)

line_kpi

Interpretation prompt:

- Which production line looks weakest on average `oee`?
- Does the weaker line also show more failure-labelled records or more defects?
- Under the current project scope, this supports line-level comparison, but not final industrial performance judgment.


### C. Simple Ranking Views

These quick ranking tables help you identify where to focus next without introducing charts yet.

In [ ]:
lowest_avg_oee_equipment = equipment_kpi.nsmallest(5, 'avg_oee')
highest_defect_equipment = equipment_kpi.nlargest(5, 'total_defect_count')
line_ranking = line_kpi.sort_values(['avg_oee', 'failure_record_count'], ascending=[True, False])

display(lowest_avg_oee_equipment)
display(highest_defect_equipment)
display(line_ranking)

Interpretation prompt:

- Do the weakest `avg_oee` equipment match the highest-defect equipment?
- Are the same lines repeatedly appearing at the weaker end of the ranking?
- These rankings are good starting points for later shift and failure analysis, but they are still descriptive, not causal.


## Shift KPI Analysis

Use this section to compare `Day`, `Evening`, and `Night` shifts.

This section answers the next practical question:

- Which shifts look weaker under the current KPI rules, and do weaker shifts also appear together with more failures or defects?


### A. Shift-Level KPI Summary

This table is the first place to compare `Day`, `Evening`, and `Night` at a high level.

In [ ]:
shift_kpi = (
    df.groupby('shift', as_index=False)
    .agg(
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        total_actual_production=('actual_production', 'sum'),
        total_defect_count=('defect_count', 'sum'),
        failure_record_count=('Machine failure', 'sum'),
    )
    .sort_values(['avg_oee', 'failure_record_count'], ascending=[True, False])
    .reset_index(drop=True)
)

shift_kpi

Interpretation prompt:

- Which shift sits at the bottom of average `oee`?
- Does the weaker shift also show more failure-labelled records or more defects?
- Under the current proxy KPI logic, this supports shift comparison, but it does not prove staffing, scheduling, or operational root cause.


### B. Production-Line + Shift Summary

This table helps answer a more practical follow-up question:

- Is a shift weak everywhere, or only weak inside specific production lines?


In [ ]:
line_shift_kpi = (
    df.groupby(['production_line', 'shift'], as_index=False)
    .agg(
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        total_actual_production=('actual_production', 'sum'),
        total_defect_count=('defect_count', 'sum'),
        failure_record_count=('Machine failure', 'sum'),
    )
    .sort_values(['avg_oee', 'failure_record_count', 'total_defect_count'], ascending=[True, False, False])
    .reset_index(drop=True)
)

line_shift_kpi

Interpretation prompt:

- Does a weak shift appear in all lines or mainly in one line?
- Which line-shift combinations deserve a closer look next?
- This is useful for targeted follow-up, but it is still descriptive under the current simulated metric design.


### C. Simple Ranking Views

These ranking tables make it easier to spot weaker shifts without adding charts yet.

In [ ]:
lowest_avg_oee_shift = shift_kpi.nsmallest(3, 'avg_oee')
highest_failure_shift = shift_kpi.nlargest(3, 'failure_record_count')
line_shift_ranking = line_shift_kpi.sort_values(['avg_oee', 'failure_record_count'], ascending=[True, False])

display(lowest_avg_oee_shift)
display(highest_failure_shift)
display(line_shift_ranking)


Interpretation prompt:

- Are the weakest `avg_oee` shifts also the same shifts with the most failure-labelled records?
- Which line-shift combinations appear repeatedly near the weaker end of the ranking?
- These rankings help prioritize deeper inspection, but they do not support industrial-grade conclusions about staffing, maintenance, or scheduling by themselves.


## Failure Summary

Use this section to inspect failure-labelled records and compare them with non-failure records.

This section answers the next practical question:

- How different do failure-labelled records look under the current proxy KPI rules, and which equipment or lines appear most often in those records?


### A. Failure vs Non-Failure Comparison

This table is the first place to compare failure-labelled and non-failure records under the current project logic.

Suggested reading habit:

- first compare `avg_oee` and `avg_quality_rate`
- then check whether failure-labelled records also show higher `avg_defect_rate` or `total_defect_count`


In [ ]:
failure_vs_non_failure_summary = (
    df.groupby('Machine failure', as_index=False)
    .agg(
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        avg_defect_rate=('defect_rate', 'mean'),
        total_actual_production=('actual_production', 'sum'),
        total_defect_count=('defect_count', 'sum'),
        record_count=('UDI', 'count'),
    )
    .sort_values('Machine failure', ascending=False)
    .reset_index(drop=True)
)

failure_vs_non_failure_summary


Interpretation prompt:

- Do failure-labelled records show lower `avg_oee`, `avg_availability`, or `avg_quality_rate` than non-failure records?
- Do they also show higher `avg_defect_rate` or more total defects?
- Under the current proxy KPI rules, this supports descriptive comparison, but it does not prove causal failure mechanisms.


### B. Equipment Failure Summary

This table helps identify which equipment IDs appear most often in failure-labelled records.


In [ ]:
equipment_failure_summary = (
    df.groupby('equipment_id', as_index=False)
    .agg(
        production_line=('production_line', 'first'),
        failure_record_count=('Machine failure', 'sum'),
        avg_oee=('oee', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        avg_defect_rate=('defect_rate', 'mean'),
        total_defect_count=('defect_count', 'sum'),
    )
    .sort_values(['failure_record_count', 'avg_oee', 'total_defect_count'], ascending=[False, True, False])
    .reset_index(drop=True)
)

equipment_failure_summary.head(10)


Interpretation prompt:

- Which equipment IDs appear most often in failure-labelled records?
- Do those same equipment IDs also show lower `avg_oee` or lower `avg_quality_rate`?
- Under the current project scope, this supports equipment prioritization, but it does not prove root cause or maintenance timing.


### C. Production-Line Failure Summary

This table rolls the same comparison up to production-line level so you can check whether failure-labelled records cluster by line.


In [ ]:
line_failure_summary = (
    df.groupby('production_line', as_index=False)
    .agg(
        failure_record_count=('Machine failure', 'sum'),
        avg_oee=('oee', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        avg_defect_rate=('defect_rate', 'mean'),
        total_defect_count=('defect_count', 'sum'),
    )
    .sort_values(['failure_record_count', 'avg_oee'], ascending=[False, True])
    .reset_index(drop=True)
)

line_failure_summary


Interpretation prompt:

- Is failure more concentrated on one production line than others?
- Does the line with more failure-labelled records also show weaker `avg_oee` or lower `avg_quality_rate`?
- Under the current proxy KPI rules, this supports line-level comparison, but not final industrial fault attribution.


### D. Simple Ranking Views

These quick ranking tables help you decide where to focus next without introducing charts yet.


In [ ]:
highest_failure_equipment = equipment_failure_summary.nlargest(5, 'failure_record_count')
highest_failure_line = line_failure_summary.nlargest(3, 'failure_record_count')

display(failure_vs_non_failure_summary)
display(highest_failure_equipment)
display(highest_failure_line)


Interpretation prompt:

- Which equipment and lines should be checked first because they appear more often in failure-labelled records?
- Are those same rows also weaker on `avg_oee`, `avg_quality_rate`, or stronger on `avg_defect_rate`?
- These rankings support descriptive prioritization only. They do not prove industrial root cause, causality, or maintenance action correctness.


## Findings Draft

Use this section to draft short findings in plain language.

Finding 1

- Observation:
- Evidence:
- Business meaning:
- Limitation:

Finding 2

- Observation:
- Evidence:
- Business meaning:
- Limitation:

Finding 3

- Observation:
- Evidence:
- Business meaning:
- Limitation:


## Limitations

Keep these limits visible when writing conclusions:

- current OEE, quality, and production fields are proxy / simulated metrics under current project rules
- current timestamps, lines, and equipment identifiers are partly generated by the preprocessing logic
- this notebook supports structured local analysis, not industrial-grade operational claims
- later notebook and dashboard work should stay aligned with `docs/analysis_report_template.md`
